# GcslRecommender: Multi-Objective Recommendations on MovieLens 1M

**Goal-Conditioned Supervised Learning (GCSL)** lets a single trained model produce
different recommendations depending on the *goals* you specify at inference time.

**How it works**: standard recommenders drop outcome columns before training.
`GcslRecommender` keeps them as **input features**, so the model learns:

    P(positive | user, item, context, outcome_1, outcome_2, ...)

At inference, an *inference method* fills in the desired outcome values (goals).
Items whose profile is most consistent with those goals score highest.

**This notebook demonstrates**:
1. Training one model with two outcome dimensions: **rating quality** and **item novelty**
2. Swapping inference methods at will — no retraining
3. Three built-in methods: `PredefinedValue`, `MeanScalarization`, `PercentileValue`
4. Writing a **custom inference method** in ~15 lines (the extensibility pitch)

**Data**: MovieLens 1M (downloaded automatically).

## 1. Imports

In [ ]:
import logging
import urllib.request
import warnings
import zipfile
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
from pandas import DataFrame

from skrec.dataset.interactions_dataset import InteractionsDataset
from skrec.dataset.items_dataset import ItemsDataset
from skrec.estimator.classification.xgb_classifier import XGBClassifierEstimator
from skrec.recommender.gcsl.gcsl_recommender import GcslRecommender
from skrec.recommender.gcsl.inference.base_inference import BaseInference
from skrec.recommender.gcsl.inference.mean_scalarization import MeanScalarization
from skrec.recommender.gcsl.inference.percentile_value import PercentileValue
from skrec.recommender.gcsl.inference.predefined_value import PredefinedValue
from skrec.scorer.universal import UniversalScorer

# Suppress verbose INFO logs from the library
logging.basicConfig(level=logging.WARNING)
logging.disable(logging.INFO)

RAW_DIR = Path("data/movielens-1m")
RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data/gcsl")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Imports OK")

## 2. Download and Load MovieLens 1M

In [ ]:
ML1M_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
zip_path = RAW_DIR / "ml-1m.zip"

if not (RAW_DIR / "ratings.dat").exists():
    print("Downloading MovieLens 1M...")
    urllib.request.urlretrieve(ML1M_URL, zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            if name.endswith(".dat"):
                with zf.open(name) as src, open(RAW_DIR / Path(name).name, "wb") as dst:
                    dst.write(src.read())
    print("Downloaded.")
else:
    print("Already downloaded.")

ratings = pd.read_csv(
    RAW_DIR / "ratings.dat", sep="::", engine="python", names=["UserID", "MovieID", "Rating", "Timestamp"]
)
movies = pd.read_csv(
    RAW_DIR / "movies.dat", sep="::", engine="python", names=["MovieID", "Title", "Genres"], encoding="latin-1"
)
users_raw = pd.read_csv(
    RAW_DIR / "users.dat", sep="::", engine="python", names=["UserID", "Gender", "Age", "Occupation", "Zip"]
)

print(f"Ratings: {len(ratings):,}  |  Users: {ratings.UserID.nunique():,}  |  Movies: {ratings.MovieID.nunique():,}")

## 3. Feature Engineering: Two Outcome Dimensions

We create two `OUTCOME_*` columns that `GcslRecommender` will keep as input features:

| Column | Meaning | Range |
|---|---|---|
| `OUTCOME_rating` | Rating quality (rating / 5) | 0.2 -- 1.0 |
| `OUTCOME_novelty` | Item rarity: `1 - (item_count / max_count)` | 0.0 (most popular) -- 1.0 (least popular) |

At inference, we'll set these to specific *goal values* to steer recommendations
toward popular blockbusters vs. hidden gems.

> **Note**: `OUTCOME_novelty` is a static item property (same for every interaction with
> a given movie). In production you'd typically use per-interaction outcomes — e.g. dwell
> time, purchase conversion, explicit rating — that genuinely vary across user-item pairs.
> We use novelty here because the popular-vs-niche contrast makes the demo visually obvious.

In [ ]:
# Item features: genre one-hot
genre_dummies = movies.Genres.str.get_dummies(sep="|").add_prefix("genre_")
items_df = pd.concat([movies[["MovieID"]].rename(columns={"MovieID": "ITEM_ID"}), genre_dummies], axis=1)
items_df["ITEM_ID"] = items_df["ITEM_ID"].astype(str)

# User features: demographics merged into interactions
users_feat = users_raw[["UserID", "Gender", "Age", "Occupation"]].copy()
users_feat["gender_M"] = (users_feat["Gender"] == "M").astype(int)
users_feat = users_feat.rename(columns={"Age": "age", "Occupation": "occupation"})
users_feat = users_feat[["UserID", "gender_M", "age", "occupation"]]

# Item novelty: 1 - (count / max_count).  Popular = 0, niche = 1.
item_counts = ratings.groupby("MovieID").size()
item_novelty = 1.0 - (item_counts / item_counts.max())

# Build multi-outcome interactions
interactions = ratings.merge(users_feat, on="UserID", how="left")
interactions = interactions.merge(item_novelty.rename("novelty").reset_index(), on="MovieID", how="left")

interactions = pd.DataFrame(
    {
        "USER_ID": interactions["UserID"].astype(str),
        "ITEM_ID": interactions["MovieID"].astype(str),
        "OUTCOME": (interactions["Rating"] >= 4).astype(float),
        "TIMESTAMP": interactions["Timestamp"],
        "gender_M": interactions["gender_M"],
        "age": interactions["age"],
        "occupation": interactions["occupation"],
        "OUTCOME_rating": interactions["Rating"] / 5.0,
        "OUTCOME_novelty": interactions["novelty"],
    }
)

print(f"Interactions: {len(interactions):,}")
print(
    f"\nOUTCOME_rating  — min: {interactions.OUTCOME_rating.min():.2f}, "
    f"mean: {interactions.OUTCOME_rating.mean():.2f}, max: {interactions.OUTCOME_rating.max():.2f}"
)
print(
    f"OUTCOME_novelty — min: {interactions.OUTCOME_novelty.min():.2f}, "
    f"mean: {interactions.OUTCOME_novelty.mean():.2f}, max: {interactions.OUTCOME_novelty.max():.2f}"
)

## 4. Train / Test Split and Dataset Creation

In [ ]:
# Leave-last-positive-out split
interactions = interactions.sort_values(["USER_ID", "TIMESTAMP"]).reset_index(drop=True)
user_counts = interactions.groupby("USER_ID").size()
interactions = interactions[interactions["USER_ID"].isin(user_counts[user_counts >= 3].index)].reset_index(drop=True)

pos = interactions[interactions["OUTCOME"] == 1.0]
last_pos_idx = pos.groupby("USER_ID")["TIMESTAMP"].idxmax()
test_df = pos.loc[last_pos_idx].reset_index(drop=True)
train_df = interactions.drop(index=last_pos_idx).reset_index(drop=True)
test_df = test_df[test_df["USER_ID"].isin(train_df["USER_ID"].unique())].reset_index(drop=True)

train_path = str(DATA_DIR / "train_interactions.csv")
items_path = str(DATA_DIR / "items.csv")
if not Path(train_path).exists():
    train_df.drop(columns=["TIMESTAMP"]).to_csv(train_path, index=False)
if not Path(items_path).exists():
    items_df.to_csv(items_path, index=False)

interactions_ds = InteractionsDataset(data_location=train_path)
items_ds = ItemsDataset(data_location=items_path)

# Columns needed at inference time (demographics + outcome columns, no TIMESTAMP/ITEM_ID)
query_cols = ["USER_ID", "gender_M", "age", "occupation", "OUTCOME_rating", "OUTCOME_novelty"]

print(f"Train: {len(train_df):,}  |  Test: {len(test_df):,}  |  Items: {len(items_df):,}")

## 5. Train the GcslRecommender

The 3-layer stack: `XGBClassifierEstimator` -> `UniversalScorer` -> `GcslRecommender`.

We start with `PredefinedValue` but can swap the inference method after training
without retraining the model — that's the whole point.

In [ ]:
estimator = XGBClassifierEstimator(
    params={
        "n_estimators": 200,
        "max_depth": 6,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "eval_metric": "logloss",
        "random_state": 42,
        "n_jobs": -1,
    }
)
scorer = UniversalScorer(estimator)
inference = PredefinedValue({"OUTCOME_rating": 0.8, "OUTCOME_novelty": 0.5})
recommender = GcslRecommender(scorer, inference_method=inference)

print("Training...")
recommender.train(interactions_ds=interactions_ds, items_ds=items_ds)
print(f"Done. Model features: {len(estimator.feature_names)} features")

### Feature Importance: Do outcome columns matter?

If the GCSL mechanism works, `OUTCOME_rating` and `OUTCOME_novelty` should rank
among the important features — otherwise goal conditioning is a no-op.

In [ ]:
import matplotlib.pyplot as plt

importances = estimator._model.get_booster().get_score(importance_type="gain")
imp_df = pd.DataFrame.from_dict(importances, orient="index", columns=["gain"]).sort_values("gain", ascending=True)

fig, ax = plt.subplots(figsize=(8, max(4, len(imp_df) * 0.3)))
colors = ["#d62728" if f.startswith("OUTCOME_") else "#1f77b4" for f in imp_df.index]
ax.barh(imp_df.index, imp_df["gain"], color=colors)
ax.set_xlabel("Feature importance (gain)")
ax.set_title("XGBoost feature importances (red = outcome columns)")
plt.tight_layout()
plt.show()

outcome_feats = [f for f in imp_df.index if f.startswith("OUTCOME_")]
print(
    f"\nOutcome feature ranks: {', '.join(f'{f} (#{len(imp_df) - imp_df.index.tolist().index(f)})' for f in outcome_feats)}"
)

## 6. Compare Goals: Same Model, Different Recommendations

We define four goal specifications using all three built-in inference methods. Goals
stay within (or near) the training distribution so the model operates in a well-supported
region.

| Goal | Method | OUTCOME_rating | OUTCOME_novelty | What it means |
|---|---|---|---|---|
| Popular-leaning | `PercentileValue` | p75 | **p25** | Good ratings, lower novelty = popular items |
| Balanced | `PredefinedValue` | 0.80 | 0.81 | Median-ish values on both dimensions |
| Niche-leaning | `PercentileValue` | p75 | **p75** | Good ratings, higher novelty = niche items |
| Scaled mean | `MeanScalarization` | 0.7x mean | 1.2x mean | Modest rating, above-average novelty |

> **Note**: Balanced and Niche-leaning goals are close on an absolute scale (novelty
> 0.81 vs 0.91), so their top-10 lists overlap. The clearest differentiation is between
> Popular-leaning and Scaled mean — the two endpoints of the novelty spectrum.

In [ ]:
# Helpers for display
movie_title = movies.set_index(movies["MovieID"].astype(str))["Title"].to_dict()
item_pop_lookup = (
    item_counts.reset_index().assign(ITEM_ID=lambda d: d["MovieID"].astype(str)).set_index("ITEM_ID")[0].to_dict()
)

# Build query: one row per sample user with demographic features + outcome columns
last_train = train_df.sort_values("TIMESTAMP").groupby("USER_ID").last().reset_index()
sample_users = ["1", "100", "1000"]
query_df = last_train[last_train["USER_ID"].isin(sample_users)][query_cols].reset_index(drop=True)

TOP_K = 10

# All three built-in inference methods represented
goals = {
    "Popular-leaning (PercentileValue: p75 rating, p25 novelty)": PercentileValue(
        {"OUTCOME_rating": 75, "OUTCOME_novelty": 25}
    ),
    "Balanced        (PredefinedValue: 0.80 rating, 0.81 novelty)": PredefinedValue(
        {"OUTCOME_rating": 0.80, "OUTCOME_novelty": 0.81}
    ),
    "Niche-leaning   (PercentileValue: p75 rating, p75 novelty)": PercentileValue(
        {"OUTCOME_rating": 75, "OUTCOME_novelty": 75}
    ),
    "Scaled mean     (MeanScalarization: 0.7x rating, 1.2x novelty)": MeanScalarization(
        {"OUTCOME_rating": 0.7, "OUTCOME_novelty": 1.2}
    ),
}

for label, method in goals.items():
    recommender.set_inference_method(method)
    recs = recommender.recommend(interactions=query_df, top_k=TOP_K)

    # Show the actual goal values used (PredefinedValue uses .goal_values, others use .goal_values_)
    goal_vals = getattr(method, "goal_values_", None) or getattr(method, "goal_values", {})
    goal_str = ", ".join(f"{k}={v:.3f}" for k, v in goal_vals.items())
    print(f"\n{'=' * 70}")
    print(f"  {label}")
    print(f"  Goals: {goal_str}")
    print(f"{'=' * 70}")
    for i, uid in enumerate(query_df["USER_ID"].tolist()):
        print(f"\n  User {uid}:")
        for rank, item_id in enumerate(recs[i], 1):
            title = movie_title.get(item_id, item_id)[:50]
            pop = item_pop_lookup.get(item_id, 0)
            print(f"    {rank:2}. {title:<52s} ({pop:,} ratings)")

## 7. Evaluation: HR@10 (Sampled Ranking)

Does goal conditioning actually help? We evaluate the **niche-leaning** goal
(p75 rating, p75 novelty) using the same leave-last-positive-out protocol as the
ranking notebook: each user's held-out test item is ranked against 100 random
negatives. HR@10 = fraction of users whose test item lands in the top 10.

**Baseline**: a plain `RankingRecommender` with the same XGBoost + demographics +
genres features (no outcome columns) achieves HR@10 ≈ 0.18 on this dataset
(see `ranking_xgboost_movielens1m.ipynb`).

In [ ]:
# Use niche-leaning goal for evaluation
recommender.set_inference_method(PercentileValue({"OUTCOME_rating": 75, "OUTCOME_novelty": 75}))

rng = np.random.default_rng(42)
all_item_ids = scorer.item_names
known_items = set(scorer.item_names)
item_name_to_idx = {name: i for i, name in enumerate(all_item_ids)}

eval_test_df = test_df[test_df["ITEM_ID"].isin(known_items)].copy()
eval_user_ids = eval_test_df["USER_ID"].unique().tolist()

# Build query rows (last training interaction per user, demographic features + outcome cols)
last_train_all = train_df.sort_values("TIMESTAMP").groupby("USER_ID").last().reset_index()
eval_query = last_train_all[last_train_all["USER_ID"].isin(eval_user_ids)][query_cols].reset_index(drop=True)

print(f"Scoring {len(eval_query):,} users against {len(all_item_ids):,} items...")
all_scores_df = recommender.score_items(interactions=eval_query)
scores_np = all_scores_df.to_numpy()
user_order = eval_query["USER_ID"].tolist()

gt_lookup = eval_test_df.set_index("USER_ID")["ITEM_ID"].to_dict()
user_seen = train_df.groupby("USER_ID")["ITEM_ID"].apply(set).to_dict()

TOP_K, N_NEGATIVES = 10, 100
hits, ndcgs = [], []

for i, user_id in enumerate(user_order):
    test_item = gt_lookup.get(user_id)
    if test_item is None or test_item not in item_name_to_idx:
        continue
    seen = user_seen.get(user_id, set())
    unseen = all_item_ids[~np.isin(all_item_ids, list(seen))]
    neg_sample = rng.choice(unseen, size=min(N_NEGATIVES, len(unseen)), replace=False)

    candidate_idxs = [item_name_to_idx[c] for c in [test_item] + list(neg_sample) if c in item_name_to_idx]
    candidate_scores = scores_np[i, candidate_idxs]
    test_score = scores_np[i, item_name_to_idx[test_item]]
    rank = int((candidate_scores > test_score).sum()) + 1

    hits.append(1 if rank <= TOP_K else 0)
    ndcgs.append(1.0 / np.log2(rank + 1) if rank <= TOP_K else 0.0)

print(f"\n{'=' * 40}")
print(f"GCSL (p75 rating, p75 novelty)")
print(f"  HR@{TOP_K}   : {np.mean(hits):.4f}")
print(f"  NDCG@{TOP_K} : {np.mean(ndcgs):.4f}")
print(f"  Users   : {len(hits):,}")
print(f"{'=' * 40}")

## 8. Extensibility: Write a Custom Inference Method

The `BaseInference` interface has two methods: `fit()` and `transform()`.
Subclass it to implement any goal-injection strategy in ~15 lines.

**Example**: `ClippedValue` — like `PredefinedValue` but clips goals to the
observed training range so you never go out-of-distribution.

In [ ]:
class ClippedValue(BaseInference):
    """Goals are clipped to [training_min, training_max] — never out-of-distribution."""

    def __init__(self, goal_values: Dict[str, float]) -> None:
        super().__init__()
        self.goal_values = goal_values

    def fit(self, interactions_df: DataFrame, outcome_cols: List[str]) -> "ClippedValue":
        self._ranges = {
            col: (float(interactions_df[col].min()), float(interactions_df[col].max())) for col in outcome_cols
        }
        self.outcome_cols_ = outcome_cols
        self._fitted = True
        return self

    def transform(self, interactions: DataFrame) -> DataFrame:
        self._check_fitted()
        interactions = interactions.copy()
        for col in self.outcome_cols_:
            lo, hi = self._ranges[col]
            interactions[col] = np.clip(self.goal_values.get(col, (lo + hi) / 2), lo, hi)
        return interactions


# Use it — rating is in-range (0.9) but novelty is below the training min (clipped to 0.0).
# This should produce popular-leaning recommendations (low novelty = popular items).
recommender.set_inference_method(ClippedValue({"OUTCOME_rating": 0.9, "OUTCOME_novelty": -5.0}))
recs = recommender.recommend(interactions=query_df, top_k=5)

clipped_goals = recommender.inference_method._ranges
print(f"Requested: rating=0.9, novelty=-5.0")
print(
    f"Clipped:   rating=0.9 (in range), novelty=0.0 (clipped from -5.0 to min={clipped_goals['OUTCOME_novelty'][0]:.2f})"
)
print()
for i, uid in enumerate(query_df["USER_ID"].tolist()):
    titles = [movie_title.get(r, r)[:45] for r in recs[i]]
    pops = [item_pop_lookup.get(r, 0) for r in recs[i]]
    print(f"  User {uid} (avg {sum(pops) / len(pops):.0f} ratings):")
    for rank, (title, pop) in enumerate(zip(titles, pops), 1):
        print(f"    {rank}. {title:<47s} ({pop:,} ratings)")

## Summary

| What | How |
|---|---|
| **Train once** | `GcslRecommender` keeps `OUTCOME_*` columns as input features |
| **Steer at inference** | Swap `inference_method` via `set_inference_method()` — no retraining |
| **Built-in methods** | `PredefinedValue` (raw values), `MeanScalarization` (scaled training mean), `PercentileValue` (percentile of training distribution) |
| **Extend** | Subclass `BaseInference` — implement `fit()` and `transform()` |
| **OOD safety** | `PredefinedValue` and `MeanScalarization` warn when goals exceed training range; `PercentileValue` is bounded by construction |

The key insight: by treating desired outcomes as **input features** rather than labels,
the model learns `P(positive | user, item, context, goals)`. Conditioning on different
goals at inference steers the ranker — popular vs. niche, engagement vs. revenue —
without touching the trained weights.